# Bank Marketing Dataset: Data Cleaning & Feature Engineering

This notebook walks through the preprocessing steps applied to the **bank-full.csv** dataset before any modeling takes place. We'll:

1. Load and inspect the raw data
2. Check for missing values, duplicates, and data types
3. Encode the target variable
4. Engineer new features based on domain knowledge and EDA insights
5. Handle outliers carefully (without destroying business meaning)


In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



---
## 1. Loading the Dataset


In [86]:
df=pd.read_csv('bank+marketing/bank/bank-full.csv',sep=";")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [87]:
df.shape


(45211, 17)

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


### Target Variable Distribution

How balanced is our target? This directly impacts modeling — a heavily imbalanced target (which we expect here) means accuracy alone won't be a reliable metric.

In [89]:
df["y"].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

A quick summary of how many features are numeric vs. categorical.

In [90]:
print(df.dtypes.value_counts().to_string())

object    10
int64      7


### Missing Values Check

Before doing anything else, we need to confirm there are no null values hiding in the data.

In [91]:
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  ✅ No missing values found!")
else:
    print(missing[missing > 0])


  ✅ No missing values found!


### Duplicates & Summary Statistics

We also check for duplicate rows and look at the basic statistics of numeric columns. The `describe()` output helps us spot unusual ranges, skewness, or suspicious values

In [92]:
print(f"Duplicate Rows: {df.duplicated().sum():,}")
print(f"Numeric Feature Statistics:")
df.describe().round(2)

Duplicate Rows: 0
Numeric Feature Statistics:


,age,balance,day,duration,campaign,pdays,previous
count,45211.00,45211.00,45211.00,45211.00,45211.00,45211.00,45211.00
mean,40.94,1362.27,15.81,258.16,2.76,40.20,0.58
std,10.62,3044.77,8.32,257.53,3.10,100.13,2.30
min,18.00,-8019.00,1.00,0.00,1.00,-1.00,0.00
25%,33.00,72.00,8.00,103.00,1.00,-1.00,0.00
50%,39.00,448.00,16.00,180.00,2.00,-1.00,0.00
75%,48.00,1428.00,21.00,319.00,3.00,-1.00,0.00
max,95.00,102127.00,31.00,4918.00,63.00,871.00,275.00


**Good news:** The dataset is clean — no missing values and no duplicate rows. We can move straight to feature engineering.

---
## 2. Target Encoding

The target column `y` contains string values (`yes` / `no`). We convert it to binary (1/0) so that models can consume it directly.

In [93]:
df['target'] = df['y'].map({'yes': 1, 'no': 0})
df = df.drop('y', axis=1)


---
## 3. High-Conversion Month Flag

Our earlier EDA revealed that certain months — specifically **March, September, October, and December** — yielded significantly higher conversion rates compared to the rest. We capture this insight as a binary feature so the model can leverage seasonality.

In [94]:
high_conv_months = ['mar', 'sep', 'oct', 'dec']

df['high_month'] = df['month'].apply(lambda x: 1 if x in high_conv_months else 0)


---
## 4. Age Group Segmentation

Instead of treating `age` as a raw number (which may have a non-linear relationship with conversion), we segment clients into meaningful life-stage groups. This helps the model capture demographic patterns and also makes business interpretation easier.

In [95]:
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], labels=['Young (<30)', 'Adult (30-45)', 'Senior (45-60)', 'Elder (>60)'])


---
## 5. Balance Group Segmentation

Account balance is heavily right-skewed, with outliers reaching over 100k. Rather than blindly clipping these values (which would lose information about wealthy clients), we bin them into meaningful brackets. This naturally handles extreme values while preserving their business significance.

In [96]:
df['balance_group'] = pd.cut(df['balance'], bins=[-np.inf, 0, 1000, 5000, np.inf], labels=['Negative/Zero', 'Low (1-1k)', 'Medium (1k-5k)', 'High (>5k)'])


### 7 .Campaign Capping

The `campaign` feature (number of contacts during this campaign) is heavily right-skewed — most clients are contacted 1-3 times, but some up to 63. We **cap it at 10**, since anyone contacted more than 10 times is operationally in the same situation, and the marginal predictive information beyond that point is minimal.


In [97]:

# Cap extreme campaign values
df['campaign'] = df['campaign'].clip(upper=10)


---
## 8. Handling `pdays` (Contact History)

The `pdays` column records how many days since the client was last contacted in a **previous** campaign. However, `-1` means "never contacted before" — it's not a real number of days. This mixes a categorical concept into a continuous variable, which confuses models.

We fix this by:
1. Replacing the `-1` values with a large number in `pdays_cleaned` so the continuous scale makes sense
2. Dropping the original `pdays` column


In [98]:
df['pdays_cleaned'] = df['pdays'].replace(-1, df['pdays'].max() + 30)
df = df.drop('pdays', axis=1)


In [99]:
df = df.drop('duration', axis=1)


---
## 9. Final Check

A quick look at the first few rows of our engineered dataset to make sure everything looks right.

In [100]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,previous,poutcome,target,high_month,age_group,balance_group,pdays_cleaned
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,0,unknown,0,0,Senior (45-60),Medium (1k-5k),901
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,0,unknown,0,0,Adult (30-45),Low (1-1k),901
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,0,unknown,0,0,Adult (30-45),Low (1-1k),901
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,0,unknown,0,0,Senior (45-60),Medium (1k-5k),901
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,1,0,unknown,0,0,Adult (30-45),Low (1-1k),901


In [101]:
df.to_csv("processed_data.csv", index=False)